In [ ]:
from dotenv import load_dotenv
import os
from langchain.agents import create_agent
from tavily import TavilyClient
from langchain_core.tools import tool
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import json


load_dotenv()

True

In [ ]:

client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
def web_search(query: str):
    return client.search(query)

@tool
def search_web(query: str):
    "Use this to search specs that mention external sources."
    return web_search(query)

tools = [search_web]


In [3]:
import os
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY")
)

In [4]:
spec_system_prompt = """
You are a PV inverter compliance and requirements extraction agent.

Your job is to extract the requirements from documents and convert it into structured JSON.
Extract all the requirements general requirements, required documents and technical requirements.
Extract as many requirements as possible but only if its in the document
Extract all the requirements from the document do not leave aynthing

Format:
{
  "parameter_name": "concise constraint"
}

────────────────────────────
EXTRACTION RULES
────────────────────────────
- Keep values short, precise, and technical
- Convert sentences into constraints (you can search the internet if the values are not provided)
  Example:
    "must match Nepal grid standard"
    → "230V ±10%, 50Hz"  
  What not to do:
   "must match Nepal grid standard"
    → "Must match Nepal grid standard" (you should use web tool here)  

- Merge duplicates into one key if needed
- Normalize technical expressions
- Extract as much requirements as you can but only if its mentioned in the document 

────────────────────────────
TOOL USAGE RULE
────────────────────────────
You have access to a web_search tool. Keep in mind that the specs are for Nepal.
Including this info in your search might help.


You must use it when:
- A requirement references external standards
  (NEA grid code, IEC standards, Nepal Electricity Authority rules).

Example:
"The inverter THD, flicker, DC injection limits, voltage range, frequency range, power factor range and anti-islanding measures shall follow the latest Grid Code of Nepal Electricity Authority"

THEN:

1. This is NOT a single constraint.
   It represents MULTIPLE technical parameters.

2. You MUST decompose it into individual technical requirements:
   - voltage range
   - frequency range
   - THD limit
   - flicker limit
   - DC injection limit
   - power factor range
   - anti-islanding requirement

3. For EACH decomposed parameter:
   - You MUST use web_search to find the latest official NEA Grid Code or equivalent standard
   - You MUST extract ONLY explicit numerical or technical limits

4. You MUST NOT:
   - copy vague phrases like “must follow grid code”
   - assume default values
   - leave parameters unprocessed if they are implied by the grid code reference

5. If web_search does not return explicit values for a parameter:
   → leave it


STANDARD RESOLUTION RULE
────────────────────────────

If a value references a standard (IEC, IEEE, NEA, etc.), such as:

- "as per IEC 61727"
- "within IEC limits"
- "according to NEA Grid Code"

THEN:

1. This is NOT a valid final output.

2. You MUST use web_search to find the exact numerical or technical limits defined in that standard.

3. You MUST replace the reference with actual constraints.

4. The final output MUST NOT contain:
   - only standard names
   - vague references like "within IEC limits"
   - incomplete citations without values

────────────────────────────
CORRECT OUTPUT FORMAT
────────────────────────────

Example:

Input:
"DC injection shall comply with IEC 61727"

After web_search:

Output:
"dc_injection_limit": "≤ 0.5% of rated current (IEC 61727:2004)"

────────────────────────────
FAILURE CASE
────────────────────────────

If web_search does not return numeric constraints:
leave it

Always use tool BEFORE finalizing the value if uncertain.

────────────────────────────
SAFETY / CONSISTENCY RULE
────────────────────────────
- Do NOT hallucinate values
- If unknown and not found via tools → leave it
- Do NOT include assumptions without evidence

────────────────────────────
FINAL OUTPUT EXAMPLE
────────────────────────────
{
  "frequency": "50Hz ±2.5%",
  "mppt_requirement": "Must include in-house MPPT compatible with PV array"
}
"""

In [5]:

spec_agent = create_agent(
    llm,
    tools=[search_web],
    system_prompt= spec_system_prompt,
    debug=True

)

In [6]:
with open("markdowns/specifications.md", "r", encoding="utf-8") as f:
    specs = f.read()

with open("markdowns/section133.md", "r", encoding="utf-8") as f:
    section133 = f.read()
    

In [7]:
input_text = f"""

#FILE 1:
{specs}

#FILE 2:
{section133}
"""

In [8]:
result = spec_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": input_text,
        }
    ]
})

[values] {'messages': [HumanMessage(content="\n\n#FILE 1:\n## **1.4 PV Inverter or Grid Connected Inverter**\n\n## **1.4.1 General Requirements**\n\nThe PV inverter or Grid connected inverter should match voltage, frequency, phase angle and phase sequences of National Grid or Mini Grid System. The PV inverter shall also in house MPPT (Maximum Power Point Tracker), an interface between Solar PV array & the Inverter. The MPPT quality standard should match with the requirements applicable to Section 1.3.3.\n\nThe inverter generated total harmonic distortion (THD), flicker, DC injection limits, Voltage Range, Frequency Range, Power Factor Range and Anti-Islanding measures at the point of connection to the utility services should follow the latest Grid Code of Nepal Electricity Authority.\n\n## **1.4.2 Required Documents**\n\n- i. The PV Inverter test certificates from IEC accredited laboratory must be provided according to:\n  - a) IEC 61727:2004 Photovoltaic (PV) Systems: Characteristics 

In [9]:
print(result['messages'][-1].content)

{
  "ac_output_voltage_three_phase": "400V ±10% L-L",
  "ac_output_voltage_single_phase": "230V ±10% L-N",
  "output_frequency": "50Hz ±2.5%",
  "mppt_efficiency": "≥ 95%",
  "inverter_efficiency_transformer_less_up_to_5kVA": "≥ 95%",
  "inverter_efficiency_transformer_less_above_5kVA": "≥ 97%",
  "euro_efficiency_transformer_less_up_to_5kVA": "≥ 94%",
  "euro_efficiency_transformer_less_above_5kVA": "≥ 96%",
  "inverter_efficiency_transformer_topology": "≥ 90%",
  "no_load_loss_transformer_less": "< 0.5% rated power",
  "no_load_loss_transformer": "< 1.5% rated power",
  "total_harmonic_distortion_thd": "< 5% at full load (IEC 61727:2004)",
  "power_factor_nominal": "> 0.99",
  "power_factor_adjustable_range": "0.8 leading to 0.8 lagging (IEC 61727:2004)",
  "ip_protection": "IP65 (IEC 60529)",
  "built_in_meter_data_logger": "Required for system performance monitoring",
  "internal_protection": "DC reverse polarity, grid fault, lightning protection",
  "automatic_operation": "Complet

In [11]:
import json
text = result["messages"][-1].content

data = json.loads(text)


with open("outputs/requirements.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

In [12]:
import json
with open("markdowns/product_spec1.md", "r", encoding="utf-8") as f:
    product_spec1 = f.read()

with open("markdowns/product_spec2.md", "r", encoding="utf-8") as f:
    product_spec2 = f.read()

with open("outputs/requirements.json", "r", encoding="utf-8") as f:
    requirements = json.load(f)

In [13]:
from langchain_core.tools import tool

@tool
def calculator(prompt: str):
    """
    Calculate or infer a specification value from product documentation.

    Use this tool when:
    - A field in requirements_json is not directly found in the text
    - You need to infer a value using surrounding context
    - You need to resolve ambiguous specification terms

    DO NOT USE ANY ASSUMPTIONS WHILE CALCULATING VALUES

    Args:
        prompt: The command to invoke an llm to calculate values for the models. It should include the inputs values
        as well as the formulas. It should contain the instructions to calculate the value 
        and the inputs required to calculate those values for each model 

        

    Returns:
        The results of the calculations.
    """
    return llm.invoke(
        f"""
Calculate the values according to the given prompt:
/n/n
{prompt}
"""
    ).content

In [56]:
extraction_system_prompt = """
You are a photovoltaic product specification extraction agent.

INPUTS

1. requirements_json
2. product specification document

TASK

Extract ONLY the fields listed in requirements_json.
The json contains the key and its recommended value but you dont need the recommended value in the output you should 
extract the value from the json
DO NOT PUT THE RECOMMENDED VALUE IN THE VALUES OF THE OUTPUT , put it from the specification document
Use the keys in the json and then extract the values from the specification document. Don't use the values from the json at all.
You can and you must derive the values that you can using the calculator tool if possible.

Tools:
You have access to a calculator agent(llm) that you can use to derive the values. You can and you must do the calculations 
in batches. You can simply give it a string or json if you like of specs of differnt models and it will calculate the 
required values and return it to you then incorporate that info in the output. Keep in mind the model is just a fresh llm so
you have to provide the inputs as well as what to calculate and how to calculate. Calculate only the values that are applicable to the model.

Also check the formula if any values cannot be calcualted and only be approximated, leave it entirely. Remeber no calculation>bad calculation


DO NOT USE any ASSUMPTIONS / HEURISTICS while calculating values, if a value cant be calculated leave it 

OUTPUT FORMAT

{ 
  "models" : {}
  "general_fields": {},
  "models_specs": {}
}

RULES


1. Read the requirements_json first.

2. Read the product specs

3. Extract all the models mentioned in the product specs



4. Search the specification provided for evidence of each requirement.

5. Every key in the json should appear in the output

5. If a value is same for all models:

{
  "general_fields": {
      ...
  }
}

Examples:
- certifications
- manufacturer
- testing laboratory
- topology
- warranty
- IP protection if common to all models
- Rated voltage (if its same for all)

6. If a value is model-specific (like power, current,etc):

{
  "models_specs": {
      "MODEL_NAME": {
          ...
      }
  }
}


7. If the document does not explicitly provide a value and you cannot derive it:

"NOT_GIVEN"

Example:

{
  "frequency": "NOT_GIVEN"
}

8. You can use the calculator tool to derive the values if possible

9. Use exact values from the document whenever possible.

10. Output valid JSON only.



11. No explanations.

12. Dont use any assumptions during calculation or extraction. 

13. No hardcoded values calculations

14. DOnot write the tolerences of the requirements.json in the output 


Every requirement key MUST exist somewhere in the output.

If found:

{
  "frequency": "50Hz"
}

If not found (and cannot be calculated):

{
  "frequency": "NOT_GIVEN"
}


"""

In [15]:
extractor_agent = create_agent(
    llm,
    system_prompt= extraction_system_prompt,
    debug=True,
    tools = [calculator]

)

In [16]:
input_text = f"""

#Requirements json:
{requirements}
\n\n\n\n
#Prodcut specs:
{product_spec1}
"""

In [17]:
result = extractor_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": input_text,
        }
    ]
})

[values] {'messages': [HumanMessage(content='\n\n#Requirements json:\n{\'ac_output_voltage_three_phase\': \'400V ±10% L-L\', \'ac_output_voltage_single_phase\': \'230V ±10% L-N\', \'output_frequency\': \'50Hz ±2.5%\', \'mppt_efficiency\': \'≥ 95%\', \'inverter_efficiency_transformer_less_up_to_5kVA\': \'≥ 95%\', \'inverter_efficiency_transformer_less_above_5kVA\': \'≥ 97%\', \'euro_efficiency_transformer_less_up_to_5kVA\': \'≥ 94%\', \'euro_efficiency_transformer_less_above_5kVA\': \'≥ 96%\', \'inverter_efficiency_transformer_topology\': \'≥ 90%\', \'no_load_loss_transformer_less\': \'< 0.5% rated power\', \'no_load_loss_transformer\': \'< 1.5% rated power\', \'total_harmonic_distortion_thd\': \'< 5% at full load (IEC 61727:2004)\', \'power_factor_nominal\': \'> 0.99\', \'power_factor_adjustable_range\': \'0.8 leading to 0.8 lagging (IEC 61727:2004)\', \'ip_protection\': \'IP65 (IEC 60529)\', \'built_in_meter_data_logger\': \'Required for system performance monitoring\', \'internal_pro

In [18]:
print(result['messages'][-1].content)

{
  "models": {
    "CE-1P3001G-230-EU": {},
    "CE-1P5001G-230-EU": {},
    "CE-1P6001G-230-EU": {},
    "CE-1P8001G-230-EU": {},
    "CE-1P10001G-230-EU": {},
    "CE-1P13001G-230-EU": {},
    "CE-1P16001G-230-EU": {},
    "CE-1P18001G-230-EU": {},
    "CE-1P20001G-230-EU": {}
  },
  "general_fields": {
    "ac_output_voltage_three_phase": "NOT_GIVEN",
    "ac_output_voltage_single_phase": "230V",
    "output_frequency": "50Hz",
    "total_harmonic_distortion_thd": "NOT_GIVEN",
    "power_factor_nominal": ">0.99",
    "power_factor_adjustable_range": "NOT_GIVEN",
    "ip_protection": "IP67",
    "built_in_meter_data_logger": "NOT_GIVEN",
    "internal_protection": "NOT_GIVEN",
    "automatic_operation": "NOT_GIVEN",
    "cooling_system": "NOT_GIVEN",
    "warranty_period": "NOT_GIVEN",
    "label_information": "NOT_GIVEN",
    "dc_injection_limit": "NOT_GIVEN",
    "voltage_range": "NOT_GIVEN",
    "frequency_range": "NOT_GIVEN",
    "flicker_limit": "NOT_GIVEN",
    "anti_islanding

In [19]:
text = result["messages"][-1].content

data = json.loads(text)

with open("outputs/specs1.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

In [20]:

DB_PATH = "./chroma_db"

embeddings = OpenAIEmbeddings()

vectorstore = Chroma(
    persist_directory=DB_PATH,
    embedding_function=embeddings
)

In [21]:
@tool
def get_top_5(query: str):
    """
    Use this tool to retrieve missing photovoltaic specification information
    from a trusted knowledge base (RAG vector database).

    PURPOSE
    -------
    This tool is ONLY for finding external, documented information that can help
    fill missing or "NOT_GIVEN" fields in a product specification JSON.

    WHEN TO USE
    -----------
    Use this tool ONLY when:
    - A field in specs1.json is "NOT_GIVEN"
    - The missing field may be available in external documents (datasheets, standards, manuals)
    - You need contextual or regulatory information (e.g. IEC standards, NEA grid rules, IP ratings, etc.)

    DO NOT USE WHEN
    ---------------
    - The value already exists in specs1.json
    - You are guessing or inferring missing values
    - The query is unrelated to PV specifications or compliance
    - You intend to fabricate or estimate values

    INPUT
    -----
    query: A precise technical search string describing the missing parameter.
           Example:
           - "IEC 61727 frequency tolerance PV inverter"
           - "NEA Nepal grid voltage 230V tolerance"
           - "IP65 definition inverter enclosure protection"

    OUTPUT
    ------
    Returns top 5 most relevant document chunks from the vector database.
    Each result contains source text that must be treated as ground truth evidence.

    IMPORTANT RULES
    ---------------
    - You MUST NOT assume anything not present in retrieved documents
    - Always prefer retrieved evidence over general knowledge
    - If retrieval does not contain usable info → put the fields as is 
    """

    results = vectorstore.similarity_search(query, k=5)
    return results

In [22]:
filler_system_prompt = """
You are a photovoltaic (PV) specification completion agent.

You are given:
- A partially filled JSON specification (specs1.json)
{ 
  "models" : {}
  "general_fields": {},
  "models_specs": {}
}
- A list of PV inverter models inside the "models" field.
-The general_fields contain values that are same for all models and the models_specs contains values that are not same for all models
- Many fields may contain "NOT_GIVEN"
- Access to:
  1. RAG tool: get_top_5(query)
     (IMPORTANT: RAG contains ONLY specification documents for PV inverter MODELS listed in "models")
  2. calculator tool: for derivations only

────────────────────────────
MISSION
────────────────────────────
Your job is to fill missing "NOT_GIVEN" values ONLY when supported by:
- the given JSON
- RAG retrieved product specifications of the EXACT listed models
- or calculator derivation based on retrieved values

You must NEVER guess or assume any values.

────────────────────────────
CRITICAL RAG SCOPE RULE
────────────────────────────
The RAG database contains PV inverter specification documents ONLY for the models listed in the "models" field.

Therefore:
- You MUST NOT search for generic inverter knowledge
- You MUST NOT use RAG as a general knowledge or standards database
- If a value is not found in the model’s specification → treat as NOT_GIVEN

────────────────────────────
CORE WORKFLOW (STRICT ORDER)
────────────────────────────

You MUST process ALL "NOT_GIVEN" fields as follows:

STEP 1 — IDENTIFY MODEL CONTEXT
- Every query MUST be either general or model-specific
Example: 'do the models have built in data logger?'


────────────────────────────
STEP 2 — RAG RETRIEVAL (ALWAYS FIRST)
The pages extracted from rag will generally have a table and it is mostly generally the same for all the models except if stated otherwise (you have to check)





────────────────────────────
STEP 3 — OPTIONAL CALCULATION
After RAG retrieval:

If retrieved model specs provide sufficient inputs:

→ Use calculator tool

Rules:
The calculator tool is itself an llm so you have to give it a prompt to calculate the values. Do not use 
any assumptions while calculating. Also you have to tell it how to calculate the values and give the appropriate inputs.

Also check the formula if any values cannot be calcualted and only be approximated, leave it entirely. Remeber no calculation>bad calculation


DO NOT USE any ASSUMPTIONS / HEURISTICS while calculating values, if a value cant be calculated leave it 
If inputs are incomplete → DO NOT use calculator

────────────────────────────
STEP 4 — FINAL DECISION
────────────────────────────

For each "NOT_GIVEN" field:

- If RAG provides explicit model-specific value → use it
- Else if calculator can derive value → use it
- Else → keep "NOT_GIVEN"

────────────────────────────
UPDATE RULES
────────────────────────────
- Replace ONLY "NOT_GIVEN" values
- Do NOT modify existing values
- Do NOT add new keys
- Preserve original JSON structure exactly

────────────────────────────
CONFLICT RULE
────────────────────────────
If multiple valid values exist across models:

Return:
{
  "value": ["value1", "value2"],
  "sources": ["MODEL_A", "MODEL_B"]
}

Never discard valid evidence.

────────────────────────────
OUTPUT FORMAT (STRICT)
────────────────────────────

Return ONLY valid JSON:

{
  "models": {},
  "general_fields": {},
  "models_specs": {}
}

────────────────────────────
ABSOLUTE RULES
────────────────────────────
-NEVER change the schema of the json
- NEVER hallucinate values
- NEVER use external/general knowledge outside model specifications
- ALWAYS include model name in reasoning and queries when applicable
- NEVER modify non-missing values
- NEVER output explanations, markdown, or notes
- If uncertain → keep "NOT_GIVEN"
"""

In [23]:
with open("outputs/specs1.json", "r", encoding="utf-8") as f:
    specs1 = f.read()


In [24]:
filler_agent = create_agent(
    llm,
    system_prompt= filler_system_prompt,
    tools = [calculator,get_top_5],
    debug=True

)

In [25]:
input_text = f"""

#Specs json:
{specs1}

"""

In [26]:
result = filler_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": input_text,
        }
    ]
})

[values] {'messages': [HumanMessage(content='\n\n#Specs json:\n{\n  "models": {\n    "CE-1P3001G-230-EU": {},\n    "CE-1P5001G-230-EU": {},\n    "CE-1P6001G-230-EU": {},\n    "CE-1P8001G-230-EU": {},\n    "CE-1P10001G-230-EU": {},\n    "CE-1P13001G-230-EU": {},\n    "CE-1P16001G-230-EU": {},\n    "CE-1P18001G-230-EU": {},\n    "CE-1P20001G-230-EU": {}\n  },\n  "general_fields": {\n    "ac_output_voltage_three_phase": "NOT_GIVEN",\n    "ac_output_voltage_single_phase": "230V",\n    "output_frequency": "50Hz",\n    "total_harmonic_distortion_thd": "NOT_GIVEN",\n    "power_factor_nominal": ">0.99",\n    "power_factor_adjustable_range": "NOT_GIVEN",\n    "ip_protection": "IP67",\n    "built_in_meter_data_logger": "NOT_GIVEN",\n    "internal_protection": "NOT_GIVEN",\n    "automatic_operation": "NOT_GIVEN",\n    "cooling_system": "NOT_GIVEN",\n    "warranty_period": "NOT_GIVEN",\n    "label_information": "NOT_GIVEN",\n    "dc_injection_limit": "NOT_GIVEN",\n    "voltage_range": "NOT_GIVEN",

In [27]:
print(result['messages'][-1].content)

{
  "models": {
    "CE-1P3001G-230-EU": {},
    "CE-1P5001G-230-EU": {},
    "CE-1P6001G-230-EU": {},
    "CE-1P8001G-230-EU": {},
    "CE-1P10001G-230-EU": {},
    "CE-1P13001G-230-EU": {},
    "CE-1P16001G-230-EU": {},
    "CE-1P18001G-230-EU": {},
    "CE-1P20001G-230-EU": {}
  },
  "general_fields": {
    "ac_output_voltage_three_phase": "NOT_GIVEN",
    "ac_output_voltage_single_phase": "230V",
    "output_frequency": "50Hz",
    "total_harmonic_distortion_thd": "NOT_GIVEN",
    "power_factor_nominal": ">0.99",
    "power_factor_adjustable_range": "NOT_GIVEN",
    "ip_protection": "IP67",
    "built_in_meter_data_logger": "NOT_GIVEN",
    "internal_protection": "NOT_GIVEN",
    "automatic_operation": "NOT_GIVEN",
    "cooling_system": "NOT_GIVEN",
    "warranty_period": "NOT_GIVEN",
    "label_information": "NOT_GIVEN",
    "dc_injection_limit": "NOT_GIVEN",
    "voltage_range": "NOT_GIVEN",
    "frequency_range": "NOT_GIVEN",
    "flicker_limit": "NOT_GIVEN",
    "anti_islanding

In [ ]:
text = result["messages"][-1].content

data = json.loads(text)

with open("outputs/specs1.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

In [58]:
input_text = f"""

#Requirements json:
{requirements}
\n\n\n\n
#Prodcut specs:
{product_spec2}
"""

In [59]:
result = extractor_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": input_text,
        }
    ]
})

[values] {'messages': [HumanMessage(content="\n\n#Requirements json:\n{'ac_output_voltage_three_phase': '400V ±10% L-L', 'ac_output_voltage_single_phase': '230V ±10% L-N', 'output_frequency': '50Hz ±2.5%', 'mppt_efficiency': '≥ 95%', 'inverter_efficiency_transformer_less_up_to_5kVA': '≥ 95%', 'inverter_efficiency_transformer_less_above_5kVA': '≥ 97%', 'euro_efficiency_transformer_less_up_to_5kVA': '≥ 94%', 'euro_efficiency_transformer_less_above_5kVA': '≥ 96%', 'inverter_efficiency_transformer_topology': '≥ 90%', 'no_load_loss_transformer_less': '< 0.5% rated power', 'no_load_loss_transformer': '< 1.5% rated power', 'total_harmonic_distortion_thd': '< 5% at full load (IEC 61727:2004)', 'power_factor_nominal': '> 0.99', 'power_factor_adjustable_range': '0.8 leading to 0.8 lagging (IEC 61727:2004)', 'ip_protection': 'IP65 (IEC 60529)', 'built_in_meter_data_logger': 'Required for system performance monitoring', 'internal_protection': 'DC reverse polarity, grid fault, lightning protection'

In [60]:
print(result["messages"][-1].content)


{
  "models": [
    "SUN-3K-G06P3-EU-AM2",
    "SUN-3K-G06P3-EU-AM2-P1",
    "SUN-4K-G06P3-EU-AM2",
    "SUN-4K-G06P3-EU-AM2-P1",
    "SUN-5K-G06P3-EU-AM2",
    "SUN-5K-G06P3-EU-AM2-P1",
    "SUN-6K-G06P3-EU-AM2",
    "SUN-6K-G06P3-EU-AM2-P1",
    "SUN-7K-G06P3-EU-AM2",
    "SUN-7K-G06P3-EU-AM2-P1",
    "SUN-8K-G06P3-EU-AM2",
    "SUN-8K-G06P3-EU-AM2-P1",
    "SUN-9K-G06P3-EU-AM2",
    "SUN-9K-G06P3-EU-AM2-P1",
    "SUN-10K-G06P3-EU-AM2",
    "SUN-10K-G06P3-EU-AM2-P1",
    "SUN-12K-G06P3-EU-AM2",
    "SUN-12K-G06P3-EU-AM2-P1",
    "SUN-15K-G06P3-EU-AM2",
    "SUN-15K-G06P3-EU-AM2-P1"
  ],
  "general_fields": {
    "ac_output_voltage_three_phase": "400V ±10% L-L",
    "ac_output_voltage_single_phase": "230V ±10% L-N",
    "output_frequency": "50/60 Hz",
    "mppt_efficiency": "NOT_GIVEN",
    "inverter_efficiency_transformer_less_up_to_5kVA": "NOT_GIVEN",
    "inverter_efficiency_transformer_less_above_5kVA": "NOT_GIVEN",
    "euro_efficiency_transformer_less_up_to_5kVA": "NOT_GIVEN",
 

In [32]:
text = result["messages"][-1].content

data = json.loads(text)

with open("outputs/specs2.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

In [33]:
combiner_system_prompt='''
You are a photovoltaic product specification reconciliation agent.

You will be given TWO structured JSON documents containing extracted product specifications for PV inverters. Both JSONs follow the similar schema:

{
  "models": {},
  "general_fields": {},
  "models_specs": {}
}

Your task is to MERGE them into a single consistent, conflict-aware, deduplicated JSON.

────────────────────────────
PRIMARY OBJECTIVE
────────────────────────────
Create a unified specification JSON that:
1. Preserves all models across both inputs
2. Merges overlapping information
3. Resolves conflicts intelligently WITHOUT losing information
4. Removes redundancy where possible
5. Clearly preserves traceability when conflicts exist

────────────────────────────
MERGING RULES
────────────────────────────

1. MODELS MERGE
- Combine all model names from both inputs.
- Ensure no model is lost or overwritten.

2. GENERAL FIELDS RULE
- Fields that are identical across ALL models should go into:
  "general_fields"

- If a field is even slightly different across models or sources:
  → it MUST be moved to "models_specs"

3. MODELS_SPECS RULE
- Store model-specific values here.
- Structure:
  "models_specs": {
      "MODEL_NAME": {
          "field": value
      }
  }

4. CONFLICT RESOLUTION
If two sources disagree:

- Do NOT discard any value.
- Instead store both values using this format:

  {
    "value": "X",
    "source": "specs1"
  },
  {
    "value": "Y",
    "source": "specs2"
  }

If source is unknown, label as "unknown".

5. NOT GIVEN HANDLING
- If a field is missing in both inputs:
  → remove it entirely (do NOT keep "NOT_GIVEN")

- If present in one input:
  → keep it as valid value

6. NORMALIZATION
- Keep units consistent (V, Hz, W, A, °C, %)
- Do NOT change numeric values unless unit conversion is explicitly required
- Preserve original technical meaning

7. PRIORITY RULE (ONLY WHEN NO CONFLICT)
- If same field exists in both and values are identical:
  → keep one copy only

8. NO HALLUCINATION RULE
- Never infer or calculate missing values
- Never add external knowledge
- Only use provided JSON inputs

────────────────────────────
OUTPUT FORMAT (STRICT)
────────────────────────────

Return ONLY valid JSON:

{
  "models": [],
  "general_fields": {},
  "models_specs": {}
}

────────────────────────────
FINAL RULES
────────────────────────────
- No explanations
- No markdown
- No extra keys outside schema
- Must be valid JSON parsable by Python json.loads()


'''

In [34]:
combiner_agent = create_agent(
    llm,
    system_prompt= combiner_system_prompt,
    debug=True

)

In [35]:

with open("outputs/specs1.json", "r", encoding="utf-8") as f:
    specs1 = json.load(f)


with open("outputs/specs2.json", "r", encoding="utf-8") as f:
    specs2 = json.load(f)

In [36]:
input_text = f"""

#product specs 1:
{specs1}
\n\n\n\n
#prodcut specs 2:
{specs2}
"""

In [38]:
result = combiner_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": input_text,
        }
    ]
})

[values] {'messages': [HumanMessage(content="\n\n#product specs 1:\n{'models': {'CE-1P3001G-230-EU': {}, 'CE-1P5001G-230-EU': {}, 'CE-1P6001G-230-EU': {}, 'CE-1P8001G-230-EU': {}, 'CE-1P10001G-230-EU': {}, 'CE-1P13001G-230-EU': {}, 'CE-1P16001G-230-EU': {}, 'CE-1P18001G-230-EU': {}, 'CE-1P20001G-230-EU': {}}, 'general_fields': {'ac_output_voltage_three_phase': 'NOT_GIVEN', 'ac_output_voltage_single_phase': '230V', 'output_frequency': '50Hz', 'total_harmonic_distortion_thd': 'NOT_GIVEN', 'power_factor_nominal': '>0.99', 'power_factor_adjustable_range': 'NOT_GIVEN', 'ip_protection': 'IP67', 'built_in_meter_data_logger': 'NOT_GIVEN', 'internal_protection': 'NOT_GIVEN', 'automatic_operation': 'NOT_GIVEN', 'cooling_system': 'NOT_GIVEN', 'warranty_period': 'NOT_GIVEN', 'label_information': 'NOT_GIVEN', 'dc_injection_limit': 'NOT_GIVEN', 'voltage_range': 'NOT_GIVEN', 'frequency_range': 'NOT_GIVEN', 'flicker_limit': 'NOT_GIVEN', 'anti_islanding_requirement': 'NOT_GIVEN', 'power_factor_range': 

In [39]:
print(result['messages'][-1].content)

{
  "models": [
    "CE-1P3001G-230-EU",
    "CE-1P5001G-230-EU",
    "CE-1P6001G-230-EU",
    "CE-1P8001G-230-EU",
    "CE-1P10001G-230-EU",
    "CE-1P13001G-230-EU",
    "CE-1P16001G-230-EU",
    "CE-1P18001G-230-EU",
    "CE-1P20001G-230-EU",
    "SUN-3K-G06P3-EU-AM2",
    "SUN-4K-G06P3-EU-AM2",
    "SUN-5K-G06P3-EU-AM2",
    "SUN-6K-G06P3-EU-AM2",
    "SUN-7K-G06P3-EU-AM2",
    "SUN-8K-G06P3-EU-AM2",
    "SUN-9K-G06P3-EU-AM2",
    "SUN-10K-G06P3-EU-AM2",
    "SUN-12K-G06P3-EU-AM2",
    "SUN-15K-G06P3-EU-AM2",
    "SUN-3K-G06P3-EU-AM2-P1",
    "SUN-4K-G06P3-EU-AM2-P1",
    "SUN-5K-G06P3-EU-AM2-P1",
    "SUN-6K-G06P3-EU-AM2-P1",
    "SUN-7K-G06P3-EU-AM2-P1",
    "SUN-8K-G06P3-EU-AM2-P1",
    "SUN-9K-G06P3-EU-AM2-P1",
    "SUN-10K-G06P3-EU-AM2-P1",
    "SUN-12K-G06P3-EU-AM2-P1",
    "SUN-15K-G06P3-EU-AM2-P1"
  ],
  "general_fields": {},
  "models_specs": {
    "CE-1P3001G-230-EU": {
      "inverter_efficiency_transformer_less_up_to_5kVA": "NOT_GIVEN",
      "inverter_efficiency_transf

In [47]:
text = result["messages"][-1].content

data = json.loads(text)

with open("outputs/combined_specs.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

In [48]:
requirement_checker_prompt = """
You are a photovoltaic (PV) product specification compliance and checker agent.

You will be given:
1. A JSON of product specifications
2. A JSON of requirements
3. A markdown document containing the requirements (for reference and clarification)

────────────────────────────
OBJECTIVE
────────────────────────────
Generate a formal compliance report suitable for Nepal import/customs review.

Your job is to evaluate whether each product model meets the given requirements based ONLY on provided evidence.

This is NOT a simple pass/fail checker.
It is a structured regulatory compliance audit report.

────────────────────────────
PRODUCT SPECIFICATION FORMAT
────────────────────────────
{
  "models": {},
  "general_fields": {},
  "models_specs": {}
}

────────────────────────────
IMPORTANT DATA STRUCTURE NOTE
────────────────────────────

The product specification JSON MAY contain values from multiple sources.

Example:
"voltage": {
    "values": ["230V ±10%", "220V ±10%"],
    "sources": ["datasheet_v1", "test_report"]
}

OR:
"voltage": ["230V ±10%", "220V ±10%"]

This means:
→ There is NO single authoritative value
→ You MUST preserve ALL values

────────────────────────────
CORE RULES
────────────────────────────

1. NEVER assume or hallucinate values
2. NEVER simplify multi-source values into one
3. NEVER average or merge conflicting values
4. ALWAYS preserve evidence from all sources
5. ONLY mark as "NOT GIVEN" if no value exists in any source after full check

────────────────────────────
MISSING DATA RULE
────────────────────────────

If a requirement field is not found in:
- product specification JSON
- supporting documents
- test report evidence

THEN:
→ output "NOT GIVEN"

BUT:
Only after verifying all sources.

────────────────────────────
CONFLICT HANDLING RULE
────────────────────────────

If a parameter has multiple values from different sources:

YOU MUST report both the values along with the sources

Rules:
- Do NOT remove any value
- Do NOT pick a preferred value
- Only collapse values if they are EXACTLY identical

────────────────────────────
EVALUATION RULES
────────────────────────────

For each requirement:
- Compare against product specification evidence


Each requirement must be classified as:
- PASS
- FAIL
- PARTIAL
- NOT VERIFIED

────────────────────────────
OUTPUT STRUCTURE (STRICT)
────────────────────────────

# 1. Executive Summary
Brief compliance overview across all models

# 2. Product Overview
- Manufacturer (if available)
- Models included
- Scope of evaluation

# 3. Compliance Assessment Tables

## 3.1 General Requirements
Markdown table with:
- Requirement
- Expected value (regulatory reference)
- Product evidence (including multi-source values if present)
- Status (PASS / FAIL / PARTIAL / NOT VERIFIED)
- Remarks (must explain reasoning)

## 3.2 Required Documents
Same format table

## 3.3 Technical Requirements
Same format table

────────────────────────────
EVIDENCE DISPLAY RULE
────────────────────────────

When showing product evidence:
- If single value → show normally
- If multiple values → show ALL values with sources

Example:
Voltage:
- 230V ±10% (datasheet)
- 220V ±10% (test report)

────────────────────────────
FINAL MODEL-WISE DECISION
────────────────────────────

For EACH model:
- PASS / FAIL / CONDITIONAL PASS
- based strictly on evidence
- include short justification

────────────────────────────
TONE
────────────────────────────
- Formal
- Regulatory / audit style
- No assumptions
- No casual language
"""

In [50]:
requirement_checker_agent = create_agent(
    llm,
    system_prompt= requirement_checker_prompt,
    debug=True

)

In [43]:

with open("outputs/combined_specs.json", "r", encoding="utf-8") as f:
    combined_specs = json.load(f)

In [51]:
input_text = f"""

#product specs:
{combined_specs}
\n\n\n\n
#requirements json:
{requirements}
#md of requirements:
{specs}
{section133}

"""

In [52]:
result = requirement_checker_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": input_text,
        }
    ]
})

[values] {'messages': [HumanMessage(content="\n\n#product specs:\n{'models': ['CE-1P3001G-230-EU', 'CE-1P5001G-230-EU', 'CE-1P6001G-230-EU', 'CE-1P8001G-230-EU', 'CE-1P10001G-230-EU', 'CE-1P13001G-230-EU', 'CE-1P16001G-230-EU', 'CE-1P18001G-230-EU', 'CE-1P20001G-230-EU', 'SUN-3K-G06P3-EU-AM2', 'SUN-4K-G06P3-EU-AM2', 'SUN-5K-G06P3-EU-AM2', 'SUN-6K-G06P3-EU-AM2', 'SUN-7K-G06P3-EU-AM2', 'SUN-8K-G06P3-EU-AM2', 'SUN-9K-G06P3-EU-AM2', 'SUN-10K-G06P3-EU-AM2', 'SUN-12K-G06P3-EU-AM2', 'SUN-15K-G06P3-EU-AM2', 'SUN-3K-G06P3-EU-AM2-P1', 'SUN-4K-G06P3-EU-AM2-P1', 'SUN-5K-G06P3-EU-AM2-P1', 'SUN-6K-G06P3-EU-AM2-P1', 'SUN-7K-G06P3-EU-AM2-P1', 'SUN-8K-G06P3-EU-AM2-P1', 'SUN-9K-G06P3-EU-AM2-P1', 'SUN-10K-G06P3-EU-AM2-P1', 'SUN-12K-G06P3-EU-AM2-P1', 'SUN-15K-G06P3-EU-AM2-P1'], 'general_fields': {}, 'models_specs': {'CE-1P3001G-230-EU': {'inverter_efficiency_transformer_less_up_to_5kVA': 'NOT_GIVEN', 'inverter_efficiency_transformer_less_above_5kVA': 'NOT_GIVEN', 'euro_efficiency_transformer_less_up_to_5k

In [53]:
print(result['messages'][-1].content)

# 1. Executive Summary

This compliance report evaluates a portfolio of 38 photovoltaic inverter models against the Nepal Electricity Authority's regulatory requirements for PV inverters. The models include two main series: CE-series (single-phase) and SUN-series (three-phase and single-phase variants). The assessment reveals that while many technical parameters such as output voltage, frequency, power factor, IP protection, and operational features are adequately documented and meet the requirements, critical efficiency metrics (inverter efficiency, euro efficiency, MPPT efficiency) and some other parameters (e.g., DC injection limit, flicker limit, no load loss, total harmonic distortion) are consistently marked as "NOT GIVEN" across all models. Consequently, no model fully satisfies all technical requirements. The SUN-series models demonstrate more comprehensive data coverage and compliance with operational and protection features, whereas the CE-series models lack efficiency and ot

In [55]:
text = result["messages"][-1].content

with open('final_output.md', "w", encoding="utf-8") as f:
    f.write(text)